In [29]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.utils import to_categorical
import pandas as pd
import numpy as np
import zipfile
import os
import matplotlib.pyplot as plt

In [32]:

# Define the path to the zip file and the extraction directory
zip_file_path = 'content/datasets.zip'
extract_dir = 'content/louisiana_images'

# Create the extraction directory if it doesn't exist
if not os.path.exists(extract_dir):
    os.makedirs(extract_dir)

# Unzip the file
print(f"Descompactando {zip_file_path} para {extract_dir}...")
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
print("Descompactação concluída.")

Descompactando content/datasets.zip para content/louisiana_images...
Descompactação concluída.


In [37]:
train_ds = tf.data.Dataset.load(extract_dir + '/zipped_datasets_temp/train_dataset')
test_ds = tf.data.Dataset.load(extract_dir + '/zipped_datasets_temp/test_dataset')

In [38]:
for img, label in train_ds.take(1):
    print("Formato da imagem:", img.shape)
    print("Tipo da imagem:", img.dtype)
    print("Rótulo:", label.numpy())

Formato da imagem: (32, 512, 360, 3)
Tipo da imagem: <dtype: 'float32'>
Rótulo: [0 1 1 0 0 0 0 0 1 0 0 0 1 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 1]


In [39]:
for imgs, labels in train_ds.take(1):
    print("Valor mínimo:", tf.reduce_min(imgs).numpy())
    print("Valor máximo:", tf.reduce_max(imgs).numpy())

Valor mínimo: 0.0
Valor máximo: 1.0


Não é preciso criar o dataset a partir do dataframe pois o arquivo importado já é um dataset.

In [40]:
from keras.applications import MobileNet
from keras.models import Model
from keras.layers import Dense, Flatten, Dropout
from keras.optimizers import Adam
from tensorflow.keras import layers

base_model = MobileNet(
    include_top=False,
    weights='imagenet',
    input_shape=(image_size[0], image_size[1], 3),  # Seu tamanho (512, 360, 3)
    alpha=1.0,
    depth_multiplier=1,
    dropout=0.001
)

# Congela as camadas do modelo base para não serem treinadas
base_model.trainable = False

x = base_model.output

# Camadas para classificação binária
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
output_layer = layers.Dense(1, activation='sigmoid')(x)

# Cria o modelo final
model = Model(inputs=base_model.input, outputs=output_layer)

# Compila o modelo
model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

# Exibe o resumo do modelo
model.summary()

print("Modelo de Transfer Learning criado e compilado com sucesso!")

C:\Users\victo\AppData\Local\Temp\ipykernel_22768\2368273794.py:7: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNet(


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 512, 360, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 256, 180, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_bn (BatchNormalization)   │ (None, 256, 180, 32)   │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_relu (ReLU)               │ (None, 256, 180, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_1 (DepthwiseConv2D)     │ (None, 256, 180, 32)   │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_1_bn                    │ (None, 256, 180, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_1_relu (ReLU)           │ (None, 256, 180, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_1 (Conv2D)              │ (None, 256, 180, 64)   │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_1_bn                    │ (None, 256, 180, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_1_relu (ReLU)           │ (None, 256, 180, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pad_2 (ZeroPadding2D)      │ (None, 257, 181, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_2 (DepthwiseConv2D)     │ (None, 128, 90, 64)    │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_2_bn                    │ (None, 128, 90, 64)    │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_2_relu (ReLU)           │ (None, 128, 90, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_2 (Conv2D)              │ (None, 128, 90, 128)   │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_2_bn                    │ (None, 128, 90, 128)   │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_2_relu (ReLU)           │ (None, 128, 90, 128)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_3 (DepthwiseConv2D)     │ (None, 128, 90, 128)   │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_3_bn                    │ (None, 128, 90, 128)   │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_3_relu (ReLU)           │ (None, 128, 90, 128)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_3 (Conv2D)              │ (None, 128, 90, 128)   │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_3_bn                    │ (None, 128, 90, 128)   │           512 │
│ (BatchNormalization)            │                        │             

 Total params: 3,229,889 (12.32 MB)

 Trainable params: 1,025 (4.00 KB)

 Non-trainable params: 3,228,864 (12.32 MB)

Modelo de Transfer Learning criado e compilado com sucesso!


In [43]:
# cria um checkpoint para salvar os pesos do melhor modelo encontrado no treinamento
checkpointer = ModelCheckpoint(filepath='model.weights.best.keras', verbose=1, save_best_only=True)

# treina o modelo
hist = model.fit(train_ds, epochs=100,
          validation_data=test_ds,
          callbacks=[checkpointer],
          verbose=2)

Epoch 1/100

Epoch 1: val_loss improved from None to 0.66751, saving model to model.weights.best.keras

Epoch 1: finished saving model to model.weights.best.keras
9/9 - 30s - 3s/step - accuracy: 0.6222 - loss: 0.6806 - val_accuracy: 0.6154 - val_loss: 0.6675
Epoch 2/100

Epoch 2: val_loss improved from 0.66751 to 0.65443, saving model to model.weights.best.keras

Epoch 2: finished saving model to model.weights.best.keras
9/9 - 20s - 2s/step - accuracy: 0.6074 - loss: 0.6749 - val_accuracy: 0.6346 - val_loss: 0.6544
Epoch 3/100

Epoch 3: val_loss improved from 0.65443 to 0.64279, saving model to model.weights.best.keras

Epoch 3: finished saving model to model.weights.best.keras
9/9 - 20s - 2s/step - accuracy: 0.6407 - loss: 0.6479 - val_accuracy: 0.6154 - val_loss: 0.6428
Epoch 4/100

Epoch 4: val_loss improved from 0.64279 to 0.63173, saving model to model.weights.best.keras

Epoch 4: finished saving model to model.weights.best.keras
9/9 - 20s - 2s/step - accuracy: 0.6630 - loss: 0.61

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

y_true = []
y_pred = []
for images, labels in test_ds.unbatch().batch(1):
    pred = model.predict(images, verbose=0)
    y_true.append(labels.numpy()[0])
    y_pred.append(1 if pred[0][0] > 0.5 else 0)

print("Matriz de confusão:\n", confusion_matrix(y_true, y_pred))
print("\nRelatório por classe:")
print(classification_report(y_true, y_pred, target_names=['Não inundação', 'Inundação']))

Matriz de confusão:
 [[36  0]
 [ 4 12]]

Relatório por classe:
               precision    recall  f1-score   support

Não inundação       0.90      1.00      0.95        36
    Inundação       1.00      0.75      0.86        16

     accuracy                           0.92        52
    macro avg       0.95      0.88      0.90        52
 weighted avg       0.93      0.92      0.92        52

